# Model Selection + stability 
- same dataset + same feature set 
- same time-based split (train = earlier, test = later)
- cross-validation only on train split 
- later: hyperparameter tuning (RF/GB) + CV again for fair comparison
- output: a clean table like "mean ± std" for each metric

# Cell 1: Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# Cell 2: Load data

In [4]:
from pathlib import Path
import pandas as pd

if Path("data").exists():
    DATA_DIR = Path("data")
elif Path("../data").exists():
    DATA_DIR = Path("../data")
else:
    raise FileNotFoundError("Can't find 'data' folder. Check CWD.")

df = pd.read_csv(DATA_DIR / "processed" / "ev_charging_features.csv")
print("Loaded:", df.shape)
df.head(3)


Loaded: (622, 34)


,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Cost (USD),...,start_hour,end_hour,start_dayofweek,start_weekday_name,is_weekend,start_part_of_day,energy_per_session_hour,distance_per_kwh,is_flexible,flexibility_label
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,13.087717,...,0,0,0,Monday,0,Night,93.403609,4.835954,0,not_flexible
1,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,35.667270,...,2,4,0,Monday,0,Night,6.831741,3.753449,0,not_flexible
2,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,13.036239,...,3,6,0,Monday,0,Night,21.475088,2.511745,1,flexible


# Cell 3 — Define features + target (your “ablation” version)
This matches the previously used feature set: drop "is_weekend", treat "start_dayofweek" as categorical

In [5]:
FEATURES_B = [
    "Battery Capacity (kWh)", "Vehicle Age (years)",
    "State of Charge (Start %)", "Distance Driven (since last charge) (km)", "Temperature (°C)",
    "Vehicle Model", "Charging Station Location", "Charger Type", "User Type",
    "start_hour", "start_dayofweek", "start_part_of_day", "is_weekend"
]

# Ablation: remove is_weekend
FEATURES = [c for c in FEATURES_B if c != "is_weekend"]

X = df[FEATURES].copy()
y = df["is_flexible"].astype(int)

# Treat day-of-week as categorical (so it gets one-hot, not numeric scaling)
if "start_dayofweek" in X.columns:
    X["start_dayofweek"] = X["start_dayofweek"].astype("category")

print("X shape:", X.shape)
print("y mean (share flexible):", y.mean().round(3))

X shape: (622, 12)
y mean (share flexible): 0.479


## Interpretation
1) X shape: (622, 12)

You have 622 charging sessions (rows) available after selecting those columns (and before your time-mask filtering step, if you run that later).

You are using 12 input features (columns).

So your model is trying to learn:
12 inputs → predict is_flexible (0/1).

2) y mean (share flexible): 0.479

- Your target y is binary (0/1), and the mean is the fraction of 1s.

- So about 47.9% of your sessions are labeled “flexible”.

- This means your classes are pretty balanced (not like 90/10).
- That’s good because accuracy won’t be “fake high” just by predicting the majority class.

3) What it doesn't tell:
- model performance 
- if the labels are meaningful (only that they exist and the split is roughly balanced) 

# Cell 4 - Time-based holdout split (past train, future test)
- This keeps my "realistic" evaluation: train on earlier sessions, test on later sessions

In [6]:
df["Charging Start Time"] = pd.to_datetime(df["Charging Start Time"], errors="coerce")
mask = df["Charging Start Time"].notna()

# Filter X/y to rows with valid timestamps
X = X.loc[mask].copy()
y = y.loc[mask].copy()

# Sort by time
order = df.loc[mask, "Charging Start Time"].sort_values().index
X = X.loc[order].reset_index(drop=True)
y = y.loc[order].reset_index(drop=True)

# 80/20 chronological split
split = int(len(X) * 0.8)
X_train, y_train = X.iloc[:split].copy(), y.iloc[:split].copy()
X_test,  y_test  = X.iloc[split:].copy(), y.iloc[split:].copy()

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))
print("Test class balance:\n", y_test.value_counts(normalize=True).round(3))

Train: (497, 12) | Test: (125, 12)
Train class balance:
 is_flexible
0    0.525
1    0.475
Name: proportion, dtype: float64
Test class balance:
 is_flexible
0    0.504
1    0.496
Name: proportion, dtype: float64


## Interpretation

1) Train: (497, 12) | Test: (125, 12)
- after removing rows with missing "Charging Start Time", 622 sessions remain in total
- I split them chronologically:
    - 497 sessions (80%) = training (earlier in time)
    - 125 sessions (20%) = test (later in time)

2) Train class balance
- In the training period, 47.5% are flexible 

3) Test class balance 
- In the test period, 49.6% are flexible

✅ So the label distribution is stable over time (no big drift like 80/20 → 20/80)

# Cell 5 - Preprocess (numeric vs categorical)

In [7]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    sparse_threshold=0.0,
)

print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

Numeric cols: ['Battery Capacity (kWh)', 'Vehicle Age (years)', 'State of Charge (Start %)', 'Distance Driven (since last charge) (km)', 'Temperature (°C)', 'start_hour']
Categorical cols: ['Vehicle Model', 'Charging Station Location', 'Charger Type', 'User Type', 'start_dayofweek', 'start_part_of_day']


# Cell 6 - Candidates (baseline + tree models)

In [8]:
candidates = {
    "LogReg": LogisticRegression(
        max_iter=5000,
        solver="saga",
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=600,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "GradBoost": GradientBoostingClassifier(random_state=42),
}

# Cell 7 — Cross-validation on TRAIN only (stability)

In [11]:
from sklearn.model_selection import TimeSeriesSplit

# X_train is already sorted by time in your earlier cell ✅
cv = TimeSeriesSplit(n_splits=5)

scoring = {
    "balanced_acc": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("preprocess", preprocess), ("clf", clf)])
    out = cross_validate(
        pipe,
        X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score=np.nan,
    )
    rows.append({
        "model": name,
        "bal_acc_mean": np.mean(out["test_balanced_acc"]),
        "bal_acc_std":  np.std(out["test_balanced_acc"]),
        "roc_auc_mean": np.mean(out["test_roc_auc"]),
        "roc_auc_std":  np.std(out["test_roc_auc"]),
        "f1_mean":      np.mean(out["test_f1"]),
        "f1_std":       np.std(out["test_f1"]),
        "precision_mean": np.mean(out["test_precision"]),
        "precision_std":  np.std(out["test_precision"]),
        "recall_mean":    np.mean(out["test_recall"]),
        "recall_std":     np.std(out["test_recall"]),
    })

cv_results_time = (
    pd.DataFrame(rows)
      .sort_values(["roc_auc_mean", "bal_acc_mean"], ascending=False)
      .reset_index(drop=True)
)

cv_results_time

,model,bal_acc_mean,bal_acc_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,RandomForest,0.509952,0.033790,0.496524,0.072664,0.432134,0.035923,0.495440,0.037968,0.388667,0.056456
1,GradBoost,0.479202,0.044671,0.481024,0.041586,0.439097,0.024163,0.460835,0.017350,0.422191,0.046109
2,LogReg,0.471306,0.051558,0.468521,0.065689,0.460295,0.020375,0.456086,0.023659,0.468088,0.043166


## Interpretation

1) RF is best, but only barely:
    - Balanced_accuracy ~ 0.51
    - ROC AUC ~ 0.497
- GradBoost und LogReg are a bit worse.
- This is basically near-random Performance (random is ~0.50 for both balanced accuracy and AUC)

2) And the std values matter too:
- RF AUC std ~ 0.073 is pretty big relative to the mean — performance varies a lot across folds.
- That’s another sign the model is not reliably learning something stable.

3)  Conclusion (for now)b
With the current feature set, predicting “flexible vs not flexible” is very difficult; the models show only weak signal.

# Cell 8 - Save CV results (so my main notebook stays untouched)

In [12]:
out_path = DATA_DIR / "processed" / "cv_results_flexibility.csv"
cv_results_time.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ..\data\processed\cv_results_flexibility.csv


1) Sanity check vs a dumb baseline (important)

In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

dummy = Pipeline([
    ("preprocess", preprocess),
    ("clf", DummyClassifier(strategy="most_frequent", random_state=42))
])

out = cross_validate(dummy, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
print("Dummy bal_acc:", np.mean(out["test_balanced_acc"]))
print("Dummy roc_auc:", np.mean(out["test_roc_auc"]))

Dummy bal_acc: 0.5
Dummy roc_auc: 0.5


2) Check holdout test performance once (the real “future” test)

In [14]:
best_pipe = Pipeline([("preprocess", preprocess), ("clf", candidates["RandomForest"])])
best_pipe.fit(X_train, y_train)

pred = best_pipe.predict(X_test)
proba = best_pipe.predict_proba(X_test)[:, 1]

print("TEST balanced acc:", round(balanced_accuracy_score(y_test, pred), 3))
print("TEST ROC AUC:", round(roc_auc_score(y_test, proba), 3))


TEST balanced acc: 0.495
TEST ROC AUC: 0.503
